- 任务：
    - 单独使用transformers加载yolo模型。观察推理结果的数据格式：
        - list类型，元素是字典类型。
    - 使用git下载yolo的其他模型，对比推理结果：概率（score）
        - tiny，small
        - 下载地址：modelscope.cn + huggingface.co
            - `git clone https://huggingface.co/hustvl/yolos-tiny`
    - 完成目标侦测与标注。
        - 在视频中集成目标侦测，并且标注
        - 视频信息标注
        - 时间标注。
    - 可选：
        - 卸载英伟达显卡驱动，安装我们指定的驱动。
        - 驱动下载deepseek，下载地址
        - 看看输出提示，是否使用的是GPU。任务管理器直接观察显卡内存。

In [2]:
# 引入模块
import cv2                                             #负责图像处理，图形界面
import transformers                                    #负责模型加载与推理（算法，训练，推理，评估）
import time                                            #获取系统时间
import PIL                                             #模型推理需要pipeline

#获取摄像头
#dev_camera = cv2.VideoCapture(0)

#或者使用视频
dev_video = cv2.VideoCapture("../../../assets/videos/street_corner.mp4")

#获取视频信息
fps = int(dev_video.get(cv2.CAP_PROP_FPS))
width = int(dev_video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(dev_video.get(cv2.CAP_PROP_FRAME_HEIGHT))

#加载模型（yolos-small）
pipe_small = transformers.pipeline(task="object-detection", 
                                   model="../../../models/yolos-small",
                                   device="cuda",
                                   use_fast=True)
#加载模型（yolos-tiny）
pipe_tiny = transformers.pipeline(task="object-detection", model="../../../models/yolos-tiny", device="cuda", use_fast=True)


#使用循环 来读取视频
while True:
    #读取
    status,image=dev_video.read()
    if status:
        ##########################################################
        #2. 图像预处理
        img_pil=PIL.Image.fromarray(image)        #即格式转换
        #3. 推理(交替体验不同的模型)
        objects=pipe_small(img_pil)
        #objects=pipe_tiny(img_pil)
        
        #4. 推理结果处理
        for obj in objects:
            score = obj["score"]
            label = obj["label"]
            box   = obj["box"]

            x1 = box["xmin"]
            y1 = box["ymin"]

            x2 = box["xmax"]
            y2 = box["ymax"]

            if label in ["car","person"] and score > 0.97:
                #目标标注
                cv2.rectangle(image,pt1=(x1,y1),pt2=(x2,y2),color=(255,255,0),thickness=2)
                
                #说明
                color = (255,0,0) if label == "person" else (0,255,0)
                cv2.putText(image,f"{label}-{score:.2f}",org=(x1,y1-10),color=color,fontFace=0,fontScale=0.75,thickness=2)
        cv2.imshow("Monitor",image)
    #显示

    #暂停（人眼视觉残留），暂停的时间：一秒除以帧数；即：int(x=1000ms/fps)
        key=cv2.waitKey(int(1000/fps))
    #判断输入键值， 如果是ESC, 则退出
        if key==27:
            print("按键退出")
            break;
    else:
        print("读取失败")
        break
dev_video.release()
cv2.destroyAllWindows()

Device set to use cuda
Device set to use cuda


按键退出


## 两种模型概率的效果截图

<div style="display: flex; gap: 10px;">
  <img src="../../../assets/images/02_rendering_tiny.png" alt="效果图1" style="width: 48%;">
  <img src="../../../assets/images/03_rendering_small.png" alt="效果图2" style="width: 48%;">
</div>